# Jurimetria Estratégica: Risco, Custo, Tempo e Eficiência

Este notebook realiza o mergulho profundo nos dados para extrair os KPIs que comporão o Dashboard Executivo.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from datetime import datetime

df = pd.read_csv('../data/legal_ops_dataset.csv')
df['data_distribuicao'] = pd.to_datetime(df['data_distribuicao'])
df['data_ultima_mov'] = pd.to_datetime(df['data_ultima_mov'])

print(f"Total de Processos na Carteira: {len(df)}")
print(f"Exposure Total (Valor da Causa): R$ {df['valor_causa'].sum():,.2f}")
print(f"Total Provisionado (CPC 25): R$ {df['valor_provisao'].sum():,.2f}")

## 1. Análise de Risco (Probabilidade de Êxito por UF)
Identificar quais estados apresentam maior risco de perda (Risco Provável).

In [ ]:
risk_by_uf = df.groupby('uf')['risco_cpc25'].value_counts(normalize=True).unstack().fillna(0)
fig = px.bar(risk_by_uf, title='Mix de Risco por UF', barmode='stack')
fig.show()

# Conclusão técnica para o LinkedIn
top_risk_uf = risk_by_uf['Provável'].idxmax()
print(f"Insight: O estado de {top_risk_uf} apresenta a maior concentração de Risco Provável.")

## 2. Análise de Custo (Custo de Defesa vs. Valor da Causa)
Onde estamos gastando mais em honorários em relação ao valor em risco?

In [ ]:
df['custo_ratio'] = (df['custo_defesa'] / df['valor_causa']) * 100
avg_cost_ratio = df.groupby('tipo_acao')['custo_ratio'].mean().reset_index()

fig = px.bar(avg_cost_ratio, x='tipo_acao', y='custo_ratio', 
             title='Custo de Defesa Médio (% do Valor da Causa)',
             labels={'custo_ratio': '% Custo/Causa'})
fig.show()

## 3. Análise de Tempo (Lead Time por Comarca/UF)
Qual o tempo médio de vida de um processo?

In [ ]:
fig = px.box(df, x='uf', y='lead_time_days', color='tipo_acao', 
             title='Lead Time Processual por UF e Tipo de Ação')
fig.show()

avg_lead_time = df['lead_time_days'].mean()
print(f"Tempo Médio de Tramitação: {avg_lead_time:.0f} dias.")

## 4. Índice de Eficiência de Legal Ops
Criamos um score de 0 a 100 onde:
- Mais pontos se o custo for baixo.
- Mais pontos se o lead time for curto.
- Mais pontos se o risco for remoto.

In [ ]:
# Normalização simples
df['score_cost'] = 1 - (df['custo_defesa'] / df['custo_defesa'].max())
df['score_time'] = 1 - (df['lead_time_days'] / df['lead_time_days'].max())
df['score_risk'] = df['risco_cpc25'].map({'Remoto': 1, 'Possível': 0.5, 'Provável': 0})

df['eficiencia_score'] = (df['score_cost'] + df['score_time'] + df['score_risk']) / 3 * 100

eff_by_uf = df.groupby('uf')['eficiencia_score'].mean().sort_values(ascending=False).reset_index()
fig = px.bar(eff_by_uf, x='uf', y='eficiencia_score', color='eficiencia_score', 
             title='Ranking de Eficiência Legal por UF')
fig.show()